In [1]:
text = """Dr. Ananya Verma, a legendary Indian physicist born in Kolkata in 1960, has made significant contributions to theoretical physics. Her achievements include the discovery of the Bose–Verma condensate, formulation of quantum entanglement theorems, and resolution of the Verma–Hawking black hole paradox. Dr. Verma’s brilliance has earned her praise from fellow scientists, and her legacy continues to inspire generations of physicists worldwide."""

In [2]:
from sentence_transformers import SentenceTransformer,util
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
#chunking
sentences = sent_tokenize(text)
chunks =[' '.join(sentences[i:i+2]) for i in range(0,len(sentences),2)]
print("Auto generated chunks are:")
print(chunks)
print(len(chunks))

Auto generated chunks are:
['Dr. Ananya Verma, a legendary Indian physicist born in Kolkata in 1960, has made significant contributions to theoretical physics. Her achievements include the discovery of the Bose–Verma condensate, formulation of quantum entanglement theorems, and resolution of the Verma–Hawking black hole paradox.', 'Dr. Verma’s brilliance has earned her praise from fellow scientists, and her legacy continues to inspire generations of physicists worldwide.']
2


In [6]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2') #nvidia/Nemotron-Orchestrator-8B

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
chunk_embeddings = model.encode(chunks,convert_to_tensor=True)

In [8]:
#embed the question
question = "what are discoveries by Dr. Ananya Verma?"
question_embedding = model.encode(question,convert_to_tensor=True)

In [10]:
cosine_scores = util.cos_sim(question_embedding,chunk_embeddings)[0]
top_k = 2
top_indices = np.argsort(-cosine_scores.cpu().numpy())[:top_k]
relevangt_chunks = [chunks[i] for i in top_indices]
print(relevangt_chunks)

['Dr. Ananya Verma, a legendary Indian physicist born in Kolkata in 1960, has made significant contributions to theoretical physics. Her achievements include the discovery of the Bose–Verma condensate, formulation of quantum entanglement theorems, and resolution of the Verma–Hawking black hole paradox.', 'Dr. Verma’s brilliance has earned her praise from fellow scientists, and her legacy continues to inspire generations of physicists worldwide.']


In [11]:
context = "\n".join(relevangt_chunks)
print(context)

Dr. Ananya Verma, a legendary Indian physicist born in Kolkata in 1960, has made significant contributions to theoretical physics. Her achievements include the discovery of the Bose–Verma condensate, formulation of quantum entanglement theorems, and resolution of the Verma–Hawking black hole paradox.
Dr. Verma’s brilliance has earned her praise from fellow scientists, and her legacy continues to inspire generations of physicists worldwide.


In [12]:
final_prompt = f"""Context:{context}
          "question":{question}
          Answer :"""
print(final_prompt)

Context:Dr. Ananya Verma, a legendary Indian physicist born in Kolkata in 1960, has made significant contributions to theoretical physics. Her achievements include the discovery of the Bose–Verma condensate, formulation of quantum entanglement theorems, and resolution of the Verma–Hawking black hole paradox.
Dr. Verma’s brilliance has earned her praise from fellow scientists, and her legacy continues to inspire generations of physicists worldwide.
          "question":what are discoveries by Dr. Ananya Verma?
          Answer :


In [14]:
from dotenv import load_dotenv
import os
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [15]:
import openai
openai.api_key = OPENAI_API_KEY
client = openai.OpenAI()
response = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
      {"role":'system',"content":"you are a helpful assistant"},
      {"role":"user","content":final_prompt}
  ]
)
answer = response.choices[0].message.content.strip()
print(answer)

Dr. Ananya Verma's discoveries include the Bose–Verma condensate, quantum entanglement theorems, and the resolution of the Verma–Hawking black hole paradox. These significant contributions have solidified her place as a legendary physicist in the field of theoretical physics.


In [16]:
from datasets import load_dataset

ds = load_dataset("Anthropic/AnthropicInterviewer")

README.md: 0.00B [00:00, ?B/s]

workforce_transcripts.csv: 0.00B [00:00, ?B/s]

creatives_transcripts.csv: 0.00B [00:00, ?B/s]

scientists_transcripts.csv: 0.00B [00:00, ?B/s]

Generating workforce split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating creatives split:   0%|          | 0/125 [00:00<?, ? examples/s]

Generating scientists split:   0%|          | 0/125 [00:00<?, ? examples/s]